# arXiv S3 bulk download and benchmark

Select candidate bundles for the team's assigned 20,000 IDs, then optionally download them to shared Drive. Manifest boundaries are candidates, not proof of member or version coverage. AWS Requester Pays charges can apply. This notebook does not crawl papers.

In [ ]:
# Directory configuration
from pathlib import Path
SNAPSHOT_PATH = Path("/content/drive/MyDrive/arxiv-inputs/arxiv-assigned-20000.jsonl")
MANIFEST_PATH = Path("/content/drive/MyDrive/arxiv-s3/runs/20261102-135934/manifest.xml")
BUNDLE_DIR = Path("/content/drive/MyDrive/arxiv-s3/packages")
REPORT_DIR = Path("/content/drive/MyDrive/arxiv-nhom1/s3-download-benchmark")
LOCAL_STAGE = Path("/content/s3-bulk-stage")

In [ ]:
# Run configuration: preview by default; explicitly enable download after review.
CODE_REF = "main"  # Publish changes before using main; revision is recorded.
REPO = "https://github.com/hcmus-arxiv-nhom1/CodeRepo.git"
RUN_ID = "bulk-plan-01"  # Use a new name for each attempt.
DOWNLOAD = False
SMOKE_IDS = []  # Empty = all 20,000; otherwise use assigned IDs only.
SMOKE_BUNDLE_KEYS = []  # Optional restriction for smoke; not full coverage.
MAX_BUNDLES = 250
MAX_MANIFEST_GB = 150.0  # Estimate only: manifest size can differ from actual size.
INDEX_MEMBERS = False  # Optional extra full read after downloading.
SNAPSHOT_SHA256 = "033d7bded327d3aa1595f5502ebfc44bcfe353a89a2011b297742d287ba4de15"

## Mount Drive and check out code

Use the account with access to the shared Drive. `main` must include this notebook's implementation. No AWS call is made during preview.

In [ ]:
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")
repo_dir = Path("/content/CodeRepo")
if any(name == "src" or name.startswith("src.") for name in sys.modules):
    raise RuntimeError("Restart runtime before switching CodeRepo revisions")
if not (repo_dir / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", REPO, str(repo_dir)], check=True)
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", CODE_REF], check=True)
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
REVISION = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "boto3", "tqdm", "psutil"], check=True)
print("Code revision:", REVISION)

## Validate the full snapshot and preview candidates

The manifest must already be available at `MANIFEST_PATH`. An ID outside all boundaries blocks a full run. A smoke selection is explicitly labelled partial. The same-month assumption is not applied; matching later-month boundaries are also included.

In [ ]:
import hashlib, json, shutil, time
from collections import Counter
from dataclasses import asdict
from src import config
from src.utils.ids import iter_id_range
from src.crawler.s3_prepare import parse_manifest
from src.benchmark.s3_bulk_download import validate_snapshot, plan_candidates

selection_started = time.monotonic()
assigned = [pid for first, last in config.RANGES.values() for pid in iter_id_range(first, last, config.MONTH_END)]
if len(assigned) != 20000 or len(set(assigned)) != 20000:
    raise ValueError("Assigned ranges do not contain exactly 20000 distinct IDs")
snapshot_info = validate_snapshot(SNAPSHOT_PATH, assigned)
if snapshot_info["sha256"] != SNAPSHOT_SHA256:
    raise ValueError("Snapshot hash mismatch; use the team's assigned full snapshot")
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Prepare the arXiv manifest at {MANIFEST_PATH} before running")
manifest_bytes = MANIFEST_PATH.read_bytes()
manifest_hash = hashlib.sha256(manifest_bytes).hexdigest()
all_bundles = parse_manifest(manifest_bytes)
if len(SMOKE_IDS) != len(set(SMOKE_IDS)) or set(SMOKE_IDS) - set(assigned):
    raise ValueError("SMOKE_IDS contains duplicates or IDs outside assigned scope")
selected_ids = SMOKE_IDS or assigned
bundles, mapping, uncovered = plan_candidates(all_bundles, selected_ids)
if uncovered:
    raise ValueError(f"Manifest does not cover {len(uncovered)} selected IDs: {uncovered[:20]}")
if SMOKE_BUNDLE_KEYS:
    unknown = set(SMOKE_BUNDLE_KEYS) - {b.key for b in bundles}
    if unknown:
        raise ValueError(f"Keys not in candidate set: {sorted(unknown)}")
    bundles = [b for b in bundles if b.key in set(SMOKE_BUNDLE_KEYS)]
partial = bool(SMOKE_IDS or SMOKE_BUNDLE_KEYS)
size = sum(b.size for b in bundles)
print("Scope:", "PARTIAL SMOKE" if partial else "FULL 20000 IDS")
print("Snapshot SHA-256:", snapshot_info["sha256"], "manifest SHA-256:", manifest_hash)
print("IDs:", len(selected_ids), "candidate bundles:", len(bundles), "manifest estimate (GB):", round(size / 1e9, 3))
print("Already present:", sum((BUNDLE_DIR / Path(b.key).name).is_file() for b in bundles))
for month in sorted({b.key.split("_")[2] for b in bundles}):
    group = sorted((b for b in bundles if b.key.split("_")[2] == month), key=lambda b: int(Path(b.key).stem.rsplit("_", 1)[1]))
    print("Month", month, "first:", group[0].key, "last:", group[-1].key, "count:", len(group))
for b in bundles:
    print(b.key, b.first, b.last, b.size, "exists:", (BUNDLE_DIR / Path(b.key).name).is_file())
selection_s = time.monotonic() - selection_started
print("Selection and validation (s):", round(selection_s, 2))
if len(bundles) > MAX_BUNDLES or size > MAX_MANIFEST_GB * 1e9:
    raise ValueError("Candidate estimate exceeds configured limit; adjust after reviewing cost")

## Download and benchmark

Set `DOWNLOAD=True` only after reviewing the selection. A new `RUN_ID` creates a separate log directory. Already-present bundles are checked by MD5; they are never silently overwritten. Copy and rename on Drive FUSE do not prove durable upload. Use only one writer for the destination folder.
Resource peaks are sampled every 2 seconds per bundle: RSS covers the process tree, disk covers only the local staging directory. Peaks can miss brief spikes. Throughput can be calculated from actual bytes and download seconds. Session wall starts after preview and AWS setup.


In [ ]:
from datetime import datetime, timezone
import csv
from tqdm.auto import tqdm
from src.benchmark.s3_bulk_download import ResourceSampler, transfer_bundle

if DOWNLOAD:
    import boto3
    from google.colab import userdata
    if not RUN_ID or any(ch not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_" for ch in RUN_ID):
        raise ValueError("RUN_ID must contain only letters, numbers, '-' and '_'")
    access = userdata.get("AWS_ACCESS_KEY_ID")
    secret = userdata.get("AWS_SECRET_ACCESS_KEY")
    try:
        token = userdata.get("AWS_SESSION_TOKEN")
    except Exception:
        token = None
    if not access or not secret:
        raise ValueError("Set AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY in Colab Secrets")
    client = boto3.client("s3", aws_access_key_id=access, aws_secret_access_key=secret,
                          aws_session_token=token or None, region_name="us-east-1")
    del access, secret, token
    LOCAL_STAGE.mkdir(parents=True, exist_ok=True)
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    run_dir = REPORT_DIR / RUN_ID
    run_dir.mkdir(exist_ok=False)
    plan = {"run_id": RUN_ID, "revision": REVISION, "snapshot_sha256": snapshot_info["sha256"],
            "manifest_sha256": manifest_hash, "scope": "partial" if partial else "full",
            "ids": selected_ids, "id_candidates": mapping, "bundles": [asdict(b) for b in bundles],
            "selection_s": selection_s, "created_utc": datetime.now(timezone.utc).isoformat()}
    (run_dir / "plan.json").write_text(json.dumps(plan, indent=2), encoding="utf-8")
    started = time.monotonic()
    results = []
    def save_summary(complete):
        summary = {"complete": complete, "session_wall_s": time.monotonic() - started,
                   "selection_s": selection_s, "counts": dict(Counter(r["status"] for r in results)),
                   "results": results}
        (run_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
        if results:
            columns = sorted(set().union(*(row.keys() for row in results)))
            with (run_dir / "bundles.csv").open("w", newline="", encoding="utf-8") as stream:
                writer = csv.DictWriter(stream, fieldnames=columns)
                writer.writeheader(); writer.writerows(results)
    save_summary(False)
    with (run_dir / "events.jsonl").open("a", encoding="utf-8") as events:
        for b in tqdm(bundles, desc="Bundles", unit="bundle"):
            try:
                if not (BUNDLE_DIR / Path(b.key).name).exists() and shutil.disk_usage(LOCAL_STAGE).free < b.size * 1.2:
                    raise OSError(f"Insufficient local free space for {b.key}; manifest size is an estimate")
                with ResourceSampler(LOCAL_STAGE, interval_s=2) as sampler:
                    with tqdm(total=b.size, desc=Path(b.key).name, unit="B", unit_scale=True, leave=False) as bar:
                        result = transfer_bundle(client, b, LOCAL_STAGE, BUNDLE_DIR, progress=bar.update)
                result.update(peak_process_tree_rss_bytes=sampler.peak_rss_bytes,
                              peak_local_stage_bytes=sampler.peak_stage_bytes,
                              resource_samples=sampler.samples, sampling_interval_s=2,
                              resource_errors=sampler.errors)
            except Exception as exc:
                event = {"key": b.key, "status": "failed", "error_type": type(exc).__name__, "message": str(exc)}
                results.append(event)
                events.write(json.dumps(event) + "\n"); events.flush()
                save_summary(False)
                raise
            results.append(result)
            events.write(json.dumps(result) + "\n"); events.flush()
            save_summary(len(results) == len(bundles))
    print("Benchmark evidence:", run_dir)
else:
    print("Preview only; no S3 calls or writes to report directory.")

## Optional member coverage check

This rereads and verifies bundles and can take a long time on Drive. A missing bundle means coverage is **unknown**, not a missing paper. Found member names do not prove the correct version.

In [ ]:
if INDEX_MEMBERS:
    from src.crawler.s3_sources import index_verified_bundle
    from collections import defaultdict
    from tqdm.auto import tqdm
    found = defaultdict(list)
    absent_bundles = []
    for b in tqdm(bundles, desc="Index verified bundles"):
        path = BUNDLE_DIR / Path(b.key).name
        if not path.is_file():
            absent_bundles.append(b.key)
            continue
        try:
            members = index_verified_bundle(path, b, set(selected_ids))
        except (ValueError, OSError) as exc:
            absent_bundles.append(b.key)
            print("Bundle could not be verified/indexed:", b.key, type(exc).__name__, str(exc))
            continue
        for pid, entry in members.items():
            found[pid].append({"bundle": b.key, "member": entry["member"], "explicit_version": entry["explicit_version"]})
    unobserved = [pid for pid in selected_ids if not found[pid]]
    ambiguous = [pid for pid in selected_ids if len(found[pid]) > 1]
    print("Found IDs:", len(found), "not observed:", len(unobserved), "multiple candidates:", len(ambiguous))
    print("Missing or unverified bundles:", len(absent_bundles), "scope:", "partial" if partial else "full")
    if absent_bundles or partial:
        print("Coverage is incomplete; unobserved IDs are not confirmed missing.")
    if DOWNLOAD and "run_dir" in globals():
        report = {"scope": "partial" if partial else "full", "unavailable_bundles": absent_bundles,
                  "unobserved_ids": unobserved, "ambiguous_ids": ambiguous, "members": dict(found)}
        (run_dir / "member-index.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
else:
    print("Member inspection skipped.")